# 02 - Feature Engineering

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/churn_data.csv")

# Same TotalCharges trap as 01_explore_data.ipynb: 11 brand-new customers
# (tenure == 0) store a blank-space string instead of a real NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df.shape

(7043, 21)

## `contract_risk` -- recency/commitment proxy

Ordinal-encode `Contract`: the shorter the commitment, the easier it is to
leave *right now*, so this plays the role "days since last active" would
play in a usage-based dataset -- it's the closest thing this data has to a
recency signal. Month-to-month = 2 (highest risk), One year = 1, Two year = 0
(lowest risk).

In [2]:
contract_risk_map = {"Month-to-month": 2, "One year": 1, "Two year": 0}
df["contract_risk"] = df["Contract"].map(contract_risk_map)

df["contract_risk"].value_counts().sort_index()

contract_risk
0    1695
1    1473
2    3875
Name: count, dtype: int64

## `num_services` -- frequency/engagement proxy

Count how many of the six optional add-on services (`OnlineSecurity`,
`OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`,
`StreamingMovies`) a customer has subscribed to (`Yes`). More add-ons
roughly stands in for "how often/deeply this customer engages with the
product" -- the frequency leg of RFM, re-mapped onto subscribed services
instead of session counts.

In [3]:
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

df["num_services"].value_counts().sort_index()

num_services
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64

## `charge_trend` -- trend proxy

`MonthlyCharges - (TotalCharges / tenure)`: current monthly rate minus the
customer's *historical average* monthly rate. Positive means their bill has
been trending up recently (often an early warning sign); negative means it's
trending down.

Edge case: the same 11 `tenure == 0` rows that broke `TotalCharges` also
break this formula (division by zero). `TotalCharges` is already `NaN` for
those rows after coercion, so `charge_trend` comes out `NaN` too rather than
raising -- that's the correct behavior (there's no "trend" to measure for a
customer with zero tenure), but it means those 11 rows need an explicit
decision (impute vs. drop) before training, same as the raw `TotalCharges`
column.

In [4]:
df["charge_trend"] = df["MonthlyCharges"] - (df["TotalCharges"] / df["tenure"])

print("NaN count:", df["charge_trend"].isna().sum())
df["charge_trend"].describe()

NaN count: 11


count    7032.000000
mean       -0.001215
std         2.616165
min       -18.900000
25%        -1.160179
50%         0.000000
75%         1.147775
max        19.125000
Name: charge_trend, dtype: float64

## `tenure` and `is_electronic_check`

- `tenure` is used directly -- already a clean recency/relationship-length
  signal, no transform needed.
- `is_electronic_check`: EDA in `01_explore_data.ipynb` showed
  `PaymentMethod == "Electronic check"` churns at 45.3% vs. 15-19% for every
  other payment method -- by far the widest gap of any category in that
  column, so it's worth a dedicated binary flag rather than relying on
  one-hot encoding to surface it later.

In [5]:
df["is_electronic_check"] = (df["PaymentMethod"] == "Electronic check").astype(int)

df["is_electronic_check"].value_counts()

is_electronic_check
0    4678
1    2365
Name: count, dtype: int64

## Correlation with Churn

Encode `Churn` as 0/1 and check each new feature's correlation with it.
Sanity check per the task: `contract_risk` should show the strongest signal,
since EDA already showed a 42.7% vs. 2.8% churn-rate gap across
`Contract` values -- the widest gap found in EDA.

In [6]:
df["churn_flag"] = (df["Churn"] == "Yes").astype(int)

engineered = ["contract_risk", "num_services", "charge_trend", "tenure", "is_electronic_check"]

corr = df[engineered + ["churn_flag"]].corr()["churn_flag"].drop("churn_flag")
corr.sort_values(key=abs, ascending=False)

contract_risk          0.396713
tenure                -0.352229
is_electronic_check    0.301919
num_services          -0.087698
charge_trend           0.002160
Name: churn_flag, dtype: float64

In [7]:
top_feature = corr.abs().idxmax()
print(f"Strongest correlated feature: {top_feature} ({corr[top_feature]:.3f})")
assert top_feature == "contract_risk", "Expected contract_risk to be the strongest signal per the EDA sanity check"
print("Sanity check passed: contract_risk is the strongest signal, as expected from EDA.")

Strongest correlated feature: contract_risk (0.397)
Sanity check passed: contract_risk is the strongest signal, as expected from EDA.


## Handle missing values: `TotalCharges` (and `charge_trend`)

Fill the 11 blank `TotalCharges` values with `0`. This isn't a guess or an
imputation strategy -- every one of these rows has `tenure == 0` (brand-new
customers who haven't been billed yet), so `0` lifetime spend is their
*true* value, not a missing one. Worth being able to say plainly: **"I
filled `TotalCharges` with 0 for `tenure == 0` customers because that's
their real lifetime spend, not a missing value in the traditional sense."**

`charge_trend` inherits the same rows as `NaN` (division by `tenure == 0`),
and filling `TotalCharges` alone doesn't fix that -- `0 / 0` is still `NaN`.
Same logic applies: a brand-new customer has no billing history yet, so
there's no trend to measure. Setting `charge_trend = 0` for those rows
(neutral -- neither trending up nor down) is the documented convention
here, for the same reason as the `TotalCharges` fill.

In [8]:
# Design decision: tenure == 0 rows get TotalCharges = 0 (true value, not an
# imputation) and charge_trend = 0 (no billing history yet => no trend).
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df.loc[df["tenure"] == 0, "charge_trend"] = 0

print("TotalCharges NaNs:", df["TotalCharges"].isna().sum())
print("charge_trend NaNs:", df["charge_trend"].isna().sum())

TotalCharges NaNs: 0
charge_trend NaNs: 0


## One-hot encode remaining categoricals

Encode: `gender`, `Partner`, `Dependents`, `PhoneService`, `MultipleLines`,
`InternetService`, `PaperlessBilling`, `PaymentMethod`.

Two categorical groups are deliberately **excluded** here, each already
represented by an engineered feature above -- one-hot encoding them again
would be redundant, over-engineered signal for what's still a Day 2 feature
set:
- The 6 service columns (`OnlineSecurity`, `OnlineBackup`,
  `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) are
  already folded into `num_services`.
- `Contract` is already ordinal-encoded as `contract_risk`.

`drop_first=True` avoids the dummy-variable trap (one linearly redundant
column per encoded feature). It's not strictly necessary for a tree model
like XGBoost -- trees are robust to redundant/correlated columns -- but it
keeps the final feature count smaller with zero cost, so there's no reason
not to. Per the plan: plain one-hot is enough here, no target encoding or
embeddings needed for a Day 2 feature set.

In [9]:
onehot_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "PaperlessBilling",
    "PaymentMethod",
]

df_encoded = pd.get_dummies(df, columns=onehot_cols, drop_first=True)
df_encoded.shape

(7043, 30)

## Drop non-feature columns

- `customerID`: unique identifier, not predictive -- would only invite
  overfitting/leakage if left in.
- `Contract`: superseded by `contract_risk` (ordinal encoding of the same
  column); keeping both would be duplicate information.
- The 6 raw service columns: superseded by `num_services`, same reasoning.
- `Churn`: the raw text label -- `churn_flag` (0/1) is the actual target
  and is kept.

In [10]:
drop_cols = ["customerID", "Contract", "Churn"] + service_cols
df_model = df_encoded.drop(columns=drop_cols)

print("Final shape:", df_model.shape)
print("\nAny non-numeric columns left?")
print(df_model.dtypes[df_model.dtypes == "object"])
df_model.head()

Final shape: (7043, 21)

Any non-numeric columns left?
Series([], dtype: object)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,contract_risk,num_services,charge_trend,is_electronic_check,churn_flag,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,2,1,0.000000,1,0,False,True,False,False,True,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,1,2,1.376471,0,0,True,False,False,True,False,False,False,False,False,False,False,True
2,0,2,53.85,108.15,2,2,-0.225000,0,1,True,False,False,True,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,1,3,1.394444,0,0,True,False,False,False,True,False,False,False,False,False,False,False
4,0,2,70.70,151.65,2,0,-5.125000,1,1,False,False,False,True,False,False,True,False,True,False,True,False


## Notes / next steps

- `TotalCharges` and `charge_trend`: the 11 `tenure == 0` rows are filled
  with `0`, not imputed -- that's their true value (no billing history
  yet), a defensible design decision rather than a guess.
- One-hot encoded 8 categorical columns (`drop_first=True`); deliberately
  skipped the 6 service columns (folded into `num_services`) and `Contract`
  (already ordinal-encoded as `contract_risk`) to avoid duplicating signal
  already captured by engineered features.
- Dropped `customerID` (identifier, not predictive), raw `Contract` and the
  6 raw service columns (superseded by engineered features), and raw
  `Churn` (superseded by `churn_flag`, the actual target).
- `df_model` is now fully numeric with zero missing values -- ready for a
  train/test split and baseline model in Day 3.